# US CPI Forecast

Source: FRED, replublished Bureau of Labor Statistics data with a clean API. 

In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

CUTOFF = '2026-03-31'

def fred_csv(series_id):
    """Pull a FRED series via public CSV endpoint, no API key needed."""
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    s = pd.read_csv(url, parse_dates=['observation_date'], index_col='observation_date')
    s.columns = [series_id]
    # FRED uses '.' for missing values - coerce to numeric
    s[series_id] = pd.to_numeric(s[series_id], errors='coerce')
    return s[series_id]

# --- Pull series ---
cpi_index_s   = fred_csv('CPIAUCSL')                 # CPI-U All Items, SA, monthly
trimmed_mean  = fred_csv('TRMMEANCPIM159SFRBCLE')    # Cleveland Fed 16% trimmed mean, YoY %
median_cpi    = fred_csv('MEDCPIM159SFRBCLE')        # Cleveland Fed median CPI, YoY %
core_cpi_idx  = fred_csv('CPILFESL')                 # CPI ex food/energy, level

# --- Build dataframe ---
df_us = pd.DataFrame({
    'cpi_index':         cpi_index_s,
    'core_cpi_index':    core_cpi_idx,
    'trimmed_mean_yoy':  trimmed_mean,
    'median_yoy':        median_cpi,
}).loc[:CUTOFF]

df_us['headline_yoy'] = df_us['cpi_index'].pct_change(12, fill_method=None) * 100
df_us['core_yoy']     = df_us['core_cpi_index'].pct_change(12, fill_method=None) * 100

# --- Sanity check ---
print(f'US CPI sample: {df_us.index.min().date()} to {df_us.index.max().date()}')
print(f'n={len(df_us)} monthly observations\n')
print('Non-null counts:')
print(df_us.notna().sum())
print('\nLast 5 obs:')
print(df_us[['cpi_index', 'headline_yoy', 'core_yoy',
             'trimmed_mean_yoy', 'median_yoy']].tail().round(2))

# Verify peaks make sense
print(f"\n2022 headline peak: {df_us.loc['2022-01':'2023-06', 'headline_yoy'].max():.2f}%  (should be ~9.1%)")
print(f"Latest headline:     {df_us['headline_yoy'].dropna().iloc[-1]:.2f}%")

US CPI sample: 1947-01-01 to 2026-03-01
n=951 monthly observations

Non-null counts:
cpi_index           950
core_cpi_index      830
trimmed_mean_yoy    508
median_yoy          508
headline_yoy        938
core_yoy            818
dtype: int64

Last 5 obs:
                  cpi_index  headline_yoy  core_yoy  trimmed_mean_yoy  \
observation_date                                                        
2025-11-01           325.06          2.70      2.60              2.86   
2025-12-01           326.03          2.65      2.65              2.91   
2026-01-01           326.59          2.39      2.51              2.72   
2026-02-01           327.46          2.43      2.47              2.67   
2026-03-01           330.29          3.29      2.60              2.64   

                  median_yoy  
observation_date              
2025-11-01              3.10  
2025-12-01              3.10  
2026-01-01              2.97  
2026-02-01              2.85  
2026-03-01              2.72  

2022 headline p

In [2]:
# Project information cutoff: 2 April 2026
# Last US CPI release available before this date is the February 2026 print
# (released mid-March 2026). March 2026 print drops mid-April → after cutoff.
CUTOFF = '2026-02-28'
df_us = df_us.loc[:CUTOFF]
print(f'Truncated to {df_us.index.max().date()}, n={len(df_us)}')
print(f'Latest headline: {df_us["headline_yoy"].dropna().iloc[-1]:.2f}%')

Truncated to 2026-02-01, n=950
Latest headline: 2.43%


In [3]:
# ===== Switch to US data =====
df = df_us.copy()                    # reuse all downstream cells

# US-specific settings
TRAIN_START = '1985-01-01'           # post-Volcker stable era
SEASONAL_M  = 12                     # monthly, not quarterly
HORIZON     = 120                    # 10 years of months
FREQ        = 'ME'                   # month-end
MU_ANCHOR   = 2.0                    # Fed target, not RBA's 2.5%

print(f'Training sample: {TRAIN_START} to {CUTOFF}')
print(f'n_obs in window: {len(df.loc[TRAIN_START:CUTOFF, "headline_yoy"].dropna())}')

Training sample: 1985-01-01 to 2026-02-28
n_obs in window: 493


In [9]:
# CPI index level
fig_idx = go.Figure()
fig_idx.add_trace(go.Scatter(x=df.index, y=df['cpi_index'],
                             mode='lines', name='CPI-U All Items',
                             line=dict(color='#1f77b4', width=2)))
fig_idx.update_layout(title='US CPI Index Level (1982–84 = 100)',
                      xaxis_title='Month', yaxis_title='Index',
                      template='plotly_white', hovermode='x unified')
fig_idx.update_xaxes(range=['1947-01-01', '2026-04-01'])
fig_idx.show()

# Inflation measures YoY
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['headline_yoy'],
                         mode='lines', name='Headline CPI YoY',
                         line=dict(color='#1f77b4', width=2)))
fig.add_trace(go.Scatter(x=df.index, y=df['core_yoy'],
                         mode='lines', name='Core CPI YoY (ex food/energy)',
                         line=dict(color='#9467bd', width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=df['trimmed_mean_yoy'],
                         mode='lines', name='Trimmed mean YoY (Cleveland Fed)',
                         line=dict(color='#d62728', width=2)))
fig.add_trace(go.Scatter(x=df.index, y=df['median_yoy'],
                         mode='lines', name='Median CPI YoY (Cleveland Fed)',
                         line=dict(color='#2ca02c', width=1.5, dash='dot')))

fig.add_hline(y=2.0, line_dash='dash', line_color='grey',
              annotation_text='Fed 2% target', annotation_position='top right')

fig.update_layout(title='US Inflation Measures (data through Feb 2026)',
                  xaxis_title='Month', yaxis_title='% change YoY',
                  template='plotly_white', hovermode='x unified',
                  legend=dict(x=0.02, y=0.98))
fig.show()

In [11]:
from statsmodels.tsa.stattools import adfuller, acf, pacf
from plotly.subplots import make_subplots

TRAIN_START = '1993-01-01'
y = df.loc[TRAIN_START:CUTOFF, 'headline_yoy'].dropna()
print(f"Training sample: {y.index.min().date()} to {y.index.max().date()}, n={len(y)}\n")

# --- ADF stationarity tests ---
print('Augmented Dickey-Fuller test (H0: unit root, non-stationary)')
for label, series in [('Levels', y), ('First difference', y.diff().dropna())]:
    stat, pval, _, _, crit, _ = adfuller(series, autolag='AIC')
    verdict = 'STATIONARY' if pval < 0.05 else 'non-stationary'
    print(f'  {label:18s}: ADF={stat:7.3f}  p-value={pval:.4f}  -> {verdict}')

# --- ACF / PACF computation ---
NLAGS = 20
ci_band = 1.96 / np.sqrt(len(y))   # 95% confidence band

fig_diag = make_subplots(rows=2, cols=2,
    subplot_titles=('ACF — YoY levels', 'PACF — YoY levels',
                    'ACF — first difference', 'PACF — first difference'))

for col_idx, (label, series) in enumerate([('levels', y), ('diff', y.diff().dropna())]):
    a = acf(series, nlags=NLAGS, fft=False)
    p = pacf(series, nlags=NLAGS, method='ywm')
    lags = np.arange(len(a))
    row = col_idx + 1
    fig_diag.add_trace(go.Bar(x=lags, y=a, marker_color='#1f77b4', showlegend=False), row=row, col=1)
    fig_diag.add_trace(go.Bar(x=lags, y=p, marker_color='#d62728', showlegend=False), row=row, col=2)
    for c in (1, 2):
        fig_diag.add_hline(y=ci_band,  line_dash='dash', line_color='grey', row=row, col=c)
        fig_diag.add_hline(y=-ci_band, line_dash='dash', line_color='grey', row=row, col=c)

fig_diag.update_layout(height=600, template='plotly_white',
                       title='Diagnostics: ACF / PACF (95% CI dashed)')
fig_diag.update_xaxes(title_text='Lag (quarters)')
fig_diag.show()

Training sample: 1993-01-01 to 2026-02-01, n=397

Augmented Dickey-Fuller test (H0: unit root, non-stationary)
  Levels            : ADF= -3.531  p-value=0.0072  -> STATIONARY
  First difference  : ADF= -5.401  p-value=0.0000  -> STATIONARY


# ARIMA

In [20]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from itertools import product
import warnings
warnings.filterwarnings('ignore')

BACKTEST_END = '2014-12-31'   # leaves 10 years of monthly held-out data through 2024
train = df.loc[TRAIN_START:BACKTEST_END, 'headline_yoy'].dropna()
test  = df.loc['2015-01-01':CUTOFF,        'headline_yoy'].dropna()
print(f'Train: {train.index.min().date()} to {train.index.max().date()}, n={len(train)}')
print(f'Test:  {test.index.min().date()}  to {test.index.max().date()}, n={len(test)}')

# --- Grid search over orders, pick lowest AIC ---
# Keep seasonal grid tight — m=12 makes each fit slower than m=4
p_range, d_range, q_range = range(0, 3), range(0, 2), range(0, 3)
P_range, D_range, Q_range = range(0, 2), range(0, 2), range(0, 2)

results = []
for p, d, q, P, D, Q in product(p_range, d_range, q_range, P_range, D_range, Q_range):
    if p == 0 and q == 0 and P == 0 and Q == 0:
        continue
    try:
        m = SARIMAX(train, order=(p, d, q), seasonal_order=(P, D, Q, SEASONAL_M),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        results.append({'order': (p, d, q), 'seasonal': (P, D, Q, SEASONAL_M),
                        'aic': m.aic, 'bic': m.bic})
    except Exception:
        continue

results_df = pd.DataFrame(results).sort_values('aic').reset_index(drop=True)
print('\nTop 5 models by AIC:')
print(results_df.head().to_string(index=False))

best_order    = results_df.iloc[0]['order']
best_seasonal = results_df.iloc[0]['seasonal']
print(f'\nBest model: SARIMAX{best_order}x{best_seasonal}')

# --- Refit best, forecast over test horizon ---
best_model = SARIMAX(train, order=best_order, seasonal_order=best_seasonal,
                     enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

fc_obj = best_model.get_forecast(steps=len(test))
fc_mean = fc_obj.predicted_mean
fc_ci80 = fc_obj.conf_int(alpha=0.20)
fc_ci95 = fc_obj.conf_int(alpha=0.05)
fc_idx  = test.index

# --- Metrics ---
actual = test.values
predicted = fc_mean.values
rmse = np.sqrt(np.mean((actual - predicted)**2))
mae  = np.mean(np.abs(actual - predicted))
mape = np.mean(np.abs((actual - predicted) / actual)) * 100
print(f'\nBacktest metrics over {len(test)} months:')
print(f'  RMSE: {rmse:.3f} pp')
print(f'  MAE:  {mae:.3f} pp')
print(f'  MAPE: {mape:.1f}%')

# --- Plot ---
fig_bt = go.Figure()
fig_bt.add_trace(go.Scatter(x=train.index, y=train.values,
                            mode='lines', name=f'Train ({TRAIN_START[:4]}–{BACKTEST_END[:4]})',
                            line=dict(color='#1f77b4')))
fig_bt.add_trace(go.Scatter(x=test.index, y=test.values,
                            mode='lines', name='Actual (held-out)',
                            line=dict(color='black', width=2)))
fig_bt.add_trace(go.Scatter(x=fc_idx, y=predicted,
                            mode='lines', name='Forecast',
                            line=dict(color='#d62728', dash='dash')))
fig_bt.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                            y=list(fc_ci95.iloc[:, 1]) + list(fc_ci95.iloc[::-1, 0]),
                            fill='toself', fillcolor='rgba(214,39,40,0.15)',
                            line=dict(width=0), name='95% CI'))
fig_bt.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                            y=list(fc_ci80.iloc[:, 1]) + list(fc_ci80.iloc[::-1, 0]),
                            fill='toself', fillcolor='rgba(214,39,40,0.25)',
                            line=dict(width=0), name='80% CI'))
fig_bt.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                 annotation_text=f'Fed {MU_ANCHOR}% target')
fig_bt.update_layout(
    title=f'US Backtest: SARIMAX{best_order}x{best_seasonal} — train {TRAIN_START[:4]}-{BACKTEST_END[:4]}, '
          f'forecast 2015-{CUTOFF[:4]}',
    xaxis_title='Month', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_bt.show()

Train: 1993-01-01 to 2014-12-01, n=264
Test:  2015-01-01  to 2026-02-01, n=133

Top 5 models by AIC:
    order      seasonal       aic       bic
(1, 1, 2) (1, 0, 1, 12) 36.679098 57.759670
(2, 1, 2) (1, 0, 1, 12) 36.819374 61.413375
(2, 0, 2) (1, 0, 1, 12) 37.494019 62.116189
(1, 0, 1) (1, 0, 1, 12) 38.862752 56.470057
(0, 1, 1) (1, 0, 1, 12) 39.195662 53.265473

Best model: SARIMAX(1, 1, 2)x(1, 0, 1, 12)

Backtest metrics over 133 months:
  RMSE: 2.161 pp
  MAE:  1.409 pp
  MAPE: 254.1%


In [35]:
# --- Refit best model on full sample (post-Volcker through cutoff) ---
y_full = df.loc[TRAIN_START:CUTOFF, 'headline_yoy'].dropna()
print(f'Full training sample: {y_full.index.min().date()} to {y_full.index.max().date()}, n={len(y_full)}')

final_model = SARIMAX(y_full, order=best_order, seasonal_order=best_seasonal,
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(f'Model: SARIMAX{best_order}x{best_seasonal}, AIC={final_model.aic:.2f}')

# --- In-sample fitted values ---
arima_fitted = final_model.fittedvalues
arima_resid  = y_full - arima_fitted

# Warmup: differencing + seasonal differencing periods
warmup = max(best_order[1], best_seasonal[1] * SEASONAL_M) + 1
arima_fitted = arima_fitted.iloc[warmup:]
arima_resid  = arima_resid.iloc[warmup:]

rmse_in = np.sqrt(np.mean(arima_resid**2))
mae_in  = np.mean(np.abs(arima_resid))
print(f'In-sample fit (excl. {warmup} warmup obs): RMSE={rmse_in:.3f} pp, MAE={mae_in:.3f} pp')

# --- Forecast 120 months (10 years) ahead ---
fc_obj  = final_model.get_forecast(steps=HORIZON)
fc_mean = fc_obj.predicted_mean
fc_ci80 = fc_obj.conf_int(alpha=0.20)
fc_ci95 = fc_obj.conf_int(alpha=0.05)

fc_idx = pd.date_range(start=y_full.index[-1] + pd.offsets.MonthEnd(),
                       periods=HORIZON, freq=FREQ)
fc_mean.index = fc_idx
fc_ci80.index = fc_idx
fc_ci95.index = fc_idx

# --- Summary stats ---
print(f'\nForecast horizon: {fc_idx[0].date()} to {fc_idx[-1].date()}')
print(f'Mean YoY forecast over 10 years: {fc_mean.mean():.2f}%')
print(f'Forecast at end of horizon:      {fc_mean.iloc[-1]:.2f}%')

# --- Plot: actual + in-sample fit + forecast + CI ---
fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=y_full.index, y=y_full.values,
                            mode='lines', name='Actual',
                            line=dict(color='#1f77b4', width=2)))
fig_fc.add_trace(go.Scatter(x=arima_fitted.index, y=arima_fitted.values,
                            mode='lines', name='In-sample fit',
                            line=dict(color='#d62728', width=1.5, dash='dot')))
fig_fc.add_trace(go.Scatter(x=fc_idx, y=fc_mean.values,
                            mode='lines', name='Forecast (mean)',
                            line=dict(color='#d62728', width=2, dash='dash')))
fig_fc.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                            y=list(fc_ci95.iloc[:, 1]) + list(fc_ci95.iloc[::-1, 0]),
                            fill='toself', fillcolor='rgba(214,39,40,0.12)',
                            line=dict(width=0), name='95% CI'))
fig_fc.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                            y=list(fc_ci80.iloc[:, 1]) + list(fc_ci80.iloc[::-1, 0]),
                            fill='toself', fillcolor='rgba(214,39,40,0.22)',
                            line=dict(width=0), name='80% CI'))
fig_fc.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                 annotation_text=f'Fed {MU_ANCHOR}% target')
fig_fc.update_layout(
    title=f'US 10-year CPI YoY Forecast — SARIMAX{best_order}x{best_seasonal}',
    xaxis_title='Month', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_fc.show()

# --- Save forecast for downstream use ---
forecast_out = pd.DataFrame({
    'yoy_mean':  fc_mean.values,
    'yoy_lo80':  fc_ci80.iloc[:, 0].values,
    'yoy_hi80':  fc_ci80.iloc[:, 1].values,
    'yoy_lo95':  fc_ci95.iloc[:, 0].values,
    'yoy_hi95':  fc_ci95.iloc[:, 1].values,
    'cpi_level': level_fc.values,
}, index=fc_idx)
print('\nForecast dataframe saved as `forecast_out`. First and last rows:')
print(forecast_out.head(2).round(2))
print(forecast_out.tail(2).round(2))

Full training sample: 1993-01-01 to 2026-02-01, n=397
Model: SARIMAX(1, 1, 2)x(1, 0, 1, 12), AIC=49.83
In-sample fit (excl. 2 warmup obs): RMSE=0.248 pp, MAE=0.174 pp

Forecast horizon: 2026-02-28 to 2036-01-31
Mean YoY forecast over 10 years: 2.63%
Forecast at end of horizon:      2.57%



Forecast dataframe saved as `forecast_out`. First and last rows:
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95  cpi_level
2026-02-28      2.65      2.34      2.96      2.17      3.13     328.15
2026-03-31      2.89      2.32      3.47      2.01      3.77     329.04
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95  cpi_level
2035-12-31      2.57      0.43      4.71      -0.7      5.84     423.89
2036-01-31      2.57      0.43      4.71      -0.7      5.84     424.03


# SARIMAX

In [26]:
# ===== SARIMAX-X backtest: train through 2014, test 2015-Feb 2026 =====

# Step 1 — forecast trimmed mean separately
tm_train = df.loc[TRAIN_START:BACKTEST_END, 'trimmed_mean_yoy'].dropna()
tm_test  = df.loc['2015-01-01':CUTOFF,        'trimmed_mean_yoy'].dropna()

tm_results = []
for p, d, q in product(range(0, 3), range(0, 2), range(0, 3)):
    if p == 0 and q == 0: continue
    try:
        m = SARIMAX(tm_train, order=(p, d, q), seasonal_order=(0, 0, 1, SEASONAL_M),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        tm_results.append({'order': (p, d, q), 'aic': m.aic})
    except Exception:
        continue
tm_best_order = pd.DataFrame(tm_results).sort_values('aic').iloc[0]['order']
print(f'Trimmed mean ARIMA: {tm_best_order}x(0,0,1,{SEASONAL_M})')

tm_model = SARIMAX(tm_train, order=tm_best_order, seasonal_order=(0, 0, 1, SEASONAL_M),
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
tm_forecast = tm_model.get_forecast(steps=len(tm_test)).predicted_mean.values

# Step 2 — fit headline SARIMAX-X with trimmed mean as exog
hl_train = df.loc[TRAIN_START:BACKTEST_END, 'headline_yoy'].dropna()
hl_test  = df.loc['2015-01-01':CUTOFF,        'headline_yoy'].dropna()

# Align trimmed-mean exog with headline train index (drop any rows where TM is missing)
common_train = hl_train.index.intersection(df['trimmed_mean_yoy'].dropna().index)
hl_train = hl_train.loc[common_train]
exog_train = df.loc[common_train, 'trimmed_mean_yoy'].values.reshape(-1, 1)

# Align TM forecast length with hl_test length
common_test = hl_test.index.intersection(tm_test.index)
hl_test = hl_test.loc[common_test]
tm_forecast = tm_forecast[:len(hl_test)]

hlx_results = []
for p, d, q, P, D, Q in product(range(0, 3), range(0, 2), range(0, 3),
                                 range(0, 2), range(0, 2), range(0, 2)):
    if p == 0 and q == 0 and P == 0 and Q == 0: continue
    try:
        m = SARIMAX(hl_train, exog=exog_train, order=(p, d, q),
                    seasonal_order=(P, D, Q, SEASONAL_M),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        hlx_results.append({'order': (p, d, q), 'seasonal': (P, D, Q, SEASONAL_M),
                            'aic': m.aic})
    except Exception:
        continue
hlx_df = pd.DataFrame(hlx_results).sort_values('aic').reset_index(drop=True)
print('\nTop 5 SARIMAX-X models by AIC:')
print(hlx_df.head().to_string(index=False))

best_x_order    = hlx_df.iloc[0]['order']
best_x_seasonal = hlx_df.iloc[0]['seasonal']

# Step 3 — refit best, forecast headline using forecasted trimmed mean as exog
best_x = SARIMAX(hl_train, exog=exog_train, order=best_x_order,
                 seasonal_order=best_x_seasonal,
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

exog_test = tm_forecast.reshape(-1, 1)
fcx_obj = best_x.get_forecast(steps=len(hl_test), exog=exog_test)
fcx_mean = fcx_obj.predicted_mean
fcx_ci80 = fcx_obj.conf_int(alpha=0.20)
fcx_ci95 = fcx_obj.conf_int(alpha=0.05)

# Step 4 — metrics, compare to baseline ARIMA
actual = hl_test.values
predicted = fcx_mean.values
rmse_x = np.sqrt(np.mean((actual - predicted)**2))
mae_x  = np.mean(np.abs(actual - predicted))
print(f'\nSARIMAX-X backtest: RMSE={rmse_x:.3f} pp, MAE={mae_x:.3f} pp')
print(f'Baseline ARIMA:     RMSE={rmse:.3f} pp, MAE={mae:.3f} pp')
print(f'Improvement:        RMSE Δ={rmse - rmse_x:+.3f} pp, MAE Δ={mae - mae_x:+.3f} pp')

# Step 5 — overlay both backtests
fig_btx = go.Figure()
fig_btx.add_trace(go.Scatter(x=hl_train.index, y=hl_train.values, mode='lines',
                             name='Train', line=dict(color='#1f77b4')))
fig_btx.add_trace(go.Scatter(x=hl_test.index, y=hl_test.values, mode='lines',
                             name='Actual', line=dict(color='black', width=2)))
fig_btx.add_trace(go.Scatter(x=hl_test.index, y=predicted, mode='lines',
                             name='SARIMAX-X forecast',
                             line=dict(color='#d62728', dash='dash', width=2)))
fig_btx.add_trace(go.Scatter(x=list(hl_test.index) + list(hl_test.index[::-1]),
                             y=list(fcx_ci95.iloc[:, 1]) + list(fcx_ci95.iloc[::-1, 0]),
                             fill='toself', fillcolor='rgba(214,39,40,0.12)',
                             line=dict(width=0), name='95% CI (SARIMAX-X)'))
fig_btx.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                 annotation_text=f'Fed {MU_ANCHOR}% target')
fig_btx.update_layout(
    title=f'US backtest comparison: SARIMAX-X{best_x_order}x{best_x_seasonal} vs baseline',
    xaxis_title='Month', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_btx.show()

Trimmed mean ARIMA: (2, 1, 2)x(0,0,1,12)

Top 5 SARIMAX-X models by AIC:
    order      seasonal        aic
(1, 0, 1) (1, 0, 1, 12) -66.250109
(0, 1, 1) (1, 0, 1, 12) -65.821080
(1, 1, 2) (1, 0, 1, 12) -65.051973
(2, 0, 1) (1, 0, 1, 12) -64.849901
(1, 1, 1) (1, 0, 1, 12) -64.498032

SARIMAX-X backtest: RMSE=1.974 pp, MAE=1.356 pp
Baseline ARIMA:     RMSE=2.161 pp, MAE=1.409 pp
Improvement:        RMSE Δ=+0.187 pp, MAE Δ=+0.053 pp


In [27]:
# ===== SARIMAX-X deliverable: refit on full sample, 10-year forecast =====

# Step 1 — forecast trimmed mean over the next 120 months
tm_full = df.loc[TRAIN_START:CUTOFF, 'trimmed_mean_yoy'].dropna()
tm_final = SARIMAX(tm_full, order=tm_best_order, seasonal_order=(0, 0, 1, SEASONAL_M),
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
tm_future = tm_final.get_forecast(steps=HORIZON).predicted_mean.values
print(f'Trimmed mean 10-yr forecast: mean {tm_future.mean():.2f}%, '
      f'end-of-horizon {tm_future[-1]:.2f}%')

# Step 2 — refit headline SARIMAX-X on full sample, aligned with trimmed-mean coverage
common_full = y_full.index.intersection(df['trimmed_mean_yoy'].dropna().index)
y_full_x = y_full.loc[common_full]
exog_full = df.loc[common_full, 'trimmed_mean_yoy'].values.reshape(-1, 1)

final_x = SARIMAX(y_full_x, exog=exog_full, order=best_x_order,
                  seasonal_order=best_x_seasonal,
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(f'SARIMAX-X{best_x_order}x{best_x_seasonal}, AIC={final_x.aic:.2f}  '
      f'(baseline ARIMA AIC={final_model.aic:.2f})')

# Step 3 — in-sample fitted values
sarimax_fitted = final_x.fittedvalues
sarimax_resid  = y_full_x - sarimax_fitted

warmup_x = max(best_x_order[1], best_x_seasonal[1] * SEASONAL_M) + 1
sarimax_fitted = sarimax_fitted.iloc[warmup_x:]
sarimax_resid  = sarimax_resid.iloc[warmup_x:]

rmse_in_x = np.sqrt(np.mean(sarimax_resid**2))
mae_in_x  = np.mean(np.abs(sarimax_resid))
print(f'In-sample fit (excl. {warmup_x} warmup obs): RMSE={rmse_in_x:.3f} pp, MAE={mae_in_x:.3f} pp')
print(f'  Baseline ARIMA in-sample: RMSE={rmse_in:.3f} pp, MAE={mae_in:.3f} pp')

# Step 4 — forecast 120 months with forecasted trimmed mean as exog
exog_future = tm_future.reshape(-1, 1)
fcx_obj  = final_x.get_forecast(steps=HORIZON, exog=exog_future)
fcx_mean = fcx_obj.predicted_mean
fcx_ci80 = fcx_obj.conf_int(alpha=0.20)
fcx_ci95 = fcx_obj.conf_int(alpha=0.05)

fcx_idx = pd.date_range(start=y_full_x.index[-1] + pd.offsets.MonthEnd(),
                        periods=HORIZON, freq=FREQ)
fcx_mean.index = fcx_idx
fcx_ci80.index = fcx_idx
fcx_ci95.index = fcx_idx

# Step 5 — summary side-by-side
print(f'\n--- 10-year forecast comparison ---')
print(f'Baseline ARIMA mean:   {fc_mean.mean():.2f}%   end:{fc_mean.iloc[-1]:.2f}%')
print(f'SARIMAX-X mean:        {fcx_mean.mean():.2f}%   end:{fcx_mean.iloc[-1]:.2f}%')
print(f'Implied target return (ARIMA):    CPI+2.5% = {fc_mean.mean() + 2.5:.2f}% p.a.')
print(f'Implied target return (SARIMAX-X): CPI+2.5% = {fcx_mean.mean() + 2.5:.2f}% p.a.')

# Step 6 — plot: actual + in-sample fit + both forecasts + CI
fig_fcx = go.Figure()
fig_fcx.add_trace(go.Scatter(x=y_full_x.index, y=y_full_x.values, mode='lines',
                             name='Actual', line=dict(color='#1f77b4', width=2)))
fig_fcx.add_trace(go.Scatter(x=sarimax_fitted.index, y=sarimax_fitted.values, mode='lines',
                             name='In-sample fit',
                             line=dict(color='#d62728', width=1.5, dash='dot')))
fig_fcx.add_trace(go.Scatter(x=fcx_idx, y=fcx_mean.values, mode='lines',
                             name='SARIMAX-X (forecast)',
                             line=dict(color='#d62728', dash='dash', width=2)))
fig_fcx.add_trace(go.Scatter(x=list(fcx_idx) + list(fcx_idx[::-1]),
                             y=list(fcx_ci95.iloc[:, 1]) + list(fcx_ci95.iloc[::-1, 0]),
                             fill='toself', fillcolor='rgba(214,39,40,0.10)',
                             line=dict(width=0), name='95% CI (SARIMAX-X)'))
fig_fcx.add_trace(go.Scatter(x=list(fcx_idx) + list(fcx_idx[::-1]),
                             y=list(fcx_ci80.iloc[:, 1]) + list(fcx_ci80.iloc[::-1, 0]),
                             fill='toself', fillcolor='rgba(214,39,40,0.20)',
                             line=dict(width=0), name='80% CI (SARIMAX-X)'))
fig_fcx.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                  annotation_text=f'Fed {MU_ANCHOR}% target')
fig_fcx.update_layout(
    title=f'US 10-year CPI Forecast: SARIMAX-X{best_x_order}x{best_x_seasonal}',
    xaxis_title='Month', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_fcx.show()

# Step 7 — save SARIMAX-X forecast
forecast_x_out = pd.DataFrame({
    'yoy_mean': fcx_mean.values,
    'yoy_lo80': fcx_ci80.iloc[:, 0].values,
    'yoy_hi80': fcx_ci80.iloc[:, 1].values,
    'yoy_lo95': fcx_ci95.iloc[:, 0].values,
    'yoy_hi95': fcx_ci95.iloc[:, 1].values,
    'tm_exog':  tm_future,
}, index=fcx_idx)
print('\nSARIMAX-X forecast saved as `forecast_x_out`. First/last rows:')
print(forecast_x_out.head(2).round(2))
print(forecast_x_out.tail(2).round(2))

Trimmed mean 10-yr forecast: mean 2.53%, end-of-horizon 2.52%
SARIMAX-X(1, 0, 1)x(1, 0, 1, 12), AIC=-123.95  (baseline ARIMA AIC=49.83)
In-sample fit (excl. 1 warmup obs): RMSE=0.197 pp, MAE=0.136 pp
  Baseline ARIMA in-sample: RMSE=0.248 pp, MAE=0.174 pp

--- 10-year forecast comparison ---
Baseline ARIMA mean:   2.26%   end:2.35%
SARIMAX-X mean:        1.95%   end:1.69%
Implied target return (ARIMA):    CPI+2.5% = 4.76% p.a.
Implied target return (SARIMAX-X): CPI+2.5% = 4.45% p.a.



SARIMAX-X forecast saved as `forecast_x_out`. First/last rows:
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95  tm_exog
2026-02-28      2.53      2.28      2.78      2.15      2.92     2.66
2026-03-31      2.66      2.22      3.11      1.98      3.35     2.62
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95  tm_exog
2035-12-31      1.69      0.41      2.97     -0.26      3.65     2.52
2036-01-31      1.69      0.41      2.97     -0.27      3.64     2.52


# VECM

In [21]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen, select_order, VECM

# Build the 3-variable system, restricted to dates where all three are non-null
vec_data = df.loc[TRAIN_START:CUTOFF, ['headline_yoy', 'trimmed_mean_yoy',
                                        'median_yoy']].dropna()
print(f'VECM sample: {vec_data.index.min().date()} to {vec_data.index.max().date()}, '
      f'n={len(vec_data)}')

# --- Johansen cointegration test ---
joh = coint_johansen(vec_data, det_order=0, k_ar_diff=2)

print('\nJohansen trace test (H0: rank ≤ r vs H1: rank > r)')
print(f'{"r":>3} {"trace stat":>12} {"5% crit":>10} {"reject H0?":>12}')
for r in range(len(joh.lr1)):
    reject = 'YES' if joh.lr1[r] > joh.cvt[r, 1] else 'no'
    print(f'{r:>3} {joh.lr1[r]:>12.3f} {joh.cvt[r, 1]:>10.3f} {reject:>12}')

print('\nInterpretation: cointegrating rank = highest r where we DO NOT reject H0.')
print('  rank=0 → no cointegration (use VAR in differences)')
print('  rank=1 → one long-run relationship')
print('  rank=2 → two long-run relationships')
print('  rank=3 → fully stationary (use VAR in levels)')

# --- Lag order selection (monthly: maxlags up to 13 captures ~1 year) ---
lag_sel = select_order(vec_data, maxlags=13, deterministic='ci')
print(f'\nLag selection by AIC: {lag_sel.aic}')
print(f'Lag selection by BIC: {lag_sel.bic}')
print(f'Lag selection by HQIC: {lag_sel.hqic}')

vecm_lag = max(lag_sel.aic, 1)
print(f'\nUsing lag order: {vecm_lag}')

VECM sample: 1993-01-01 to 2026-02-01, n=397

Johansen trace test (H0: rank ≤ r vs H1: rank > r)
  r   trace stat    5% crit   reject H0?
  0       91.351     29.796          YES
  1       42.753     15.494          YES
  2       11.823      3.841          YES

Interpretation: cointegrating rank = highest r where we DO NOT reject H0.
  rank=0 → no cointegration (use VAR in differences)
  rank=1 → one long-run relationship
  rank=2 → two long-run relationships
  rank=3 → fully stationary (use VAR in levels)

Lag selection by AIC: 13
Lag selection by BIC: 1
Lag selection by HQIC: 13

Using lag order: 13


# VAR

In [23]:
from statsmodels.tsa.api import VAR

var_data = df.loc[TRAIN_START:CUTOFF,
                  ['headline_yoy', 'trimmed_mean_yoy', 'median_yoy']].dropna()
HORIZON = 120

print(f'{"Lag":>5} {"AIC":>10} {"BIC":>10} {"10-yr mean":>14} {"end-horizon":>14}')
for lag in [1, 2, 3, 6, 12]:
    m = VAR(var_data).fit(lag)
    fc = m.forecast(y=var_data.values[-lag:], steps=HORIZON)
    print(f'{lag:>5} {m.aic:>10.3f} {m.bic:>10.3f} '
          f'{fc[:, 0].mean():>13.2f}% {fc[-1, 0]:>13.2f}%')

  Lag        AIC        BIC     10-yr mean    end-horizon
    1    -12.329    -12.208          2.50%          2.52%
    2    -12.743    -12.531          2.54%          2.54%
    3    -12.839    -12.536          2.53%          2.54%
    6    -12.877    -12.299          2.58%          2.55%
   12    -12.915    -11.775          2.61%          2.52%


In [24]:
# ===== VAR backtest: train through 2014, test 2015-Feb 2026 =====
var_train = df.loc[TRAIN_START:BACKTEST_END,
                   ['headline_yoy', 'trimmed_mean_yoy', 'median_yoy']].dropna()
var_test  = df.loc['2015-01-01':CUTOFF,
                   ['headline_yoy', 'trimmed_mean_yoy', 'median_yoy']].dropna()
print(f'Train: {var_train.index.min().date()} to {var_train.index.max().date()}, n={len(var_train)}')
print(f'Test:  {var_test.index.min().date()}  to {var_test.index.max().date()},  n={len(var_test)}')

VAR_LAG = 2
var_model = VAR(var_train).fit(VAR_LAG)
print(f'VAR({VAR_LAG}) fitted, AIC={var_model.aic:.3f}')

# Forecast over test horizon
fcv_mid, fcv_lo95, fcv_hi95 = var_model.forecast_interval(
    y=var_train.values[-VAR_LAG:], steps=len(var_test), alpha=0.05)
_, fcv_lo80, fcv_hi80 = var_model.forecast_interval(
    y=var_train.values[-VAR_LAG:], steps=len(var_test), alpha=0.20)

H = 0   # headline column index
fcv_headline = fcv_mid[:, H]
fcv_h_lo95   = fcv_lo95[:, H]
fcv_h_hi95   = fcv_hi95[:, H]
fcv_h_lo80   = fcv_lo80[:, H]
fcv_h_hi80   = fcv_hi80[:, H]

# Metrics on headline
actual = var_test['headline_yoy'].values
rmse_var = np.sqrt(np.mean((actual - fcv_headline)**2))
mae_var  = np.mean(np.abs(actual - fcv_headline))
print(f'\nVAR backtest (headline only):')
print(f'  RMSE: {rmse_var:.3f} pp')
print(f'  MAE:  {mae_var:.3f} pp')
print(f'\nComparison with prior models:')
print(f'  Baseline ARIMA: RMSE={rmse:.3f}, MAE={mae:.3f}')
print(f'  SARIMAX-X:      RMSE={rmse_x:.3f}, MAE={mae_x:.3f}')
print(f'  VAR({VAR_LAG}):          RMSE={rmse_var:.3f}, MAE={mae_var:.3f}')

# Plot
fig_btv = go.Figure()
fig_btv.add_trace(go.Scatter(x=var_train.index, y=var_train['headline_yoy'].values,
                             mode='lines', name='Train', line=dict(color='#1f77b4')))
fig_btv.add_trace(go.Scatter(x=var_test.index, y=actual, mode='lines',
                             name='Actual', line=dict(color='black', width=2)))
fig_btv.add_trace(go.Scatter(x=var_test.index, y=fcv_headline, mode='lines',
                             name=f'VAR({VAR_LAG}) forecast',
                             line=dict(color='#9467bd', dash='dash', width=2)))
fig_btv.add_trace(go.Scatter(x=list(var_test.index) + list(var_test.index[::-1]),
                             y=list(fcv_h_hi95) + list(fcv_h_lo95[::-1]),
                             fill='toself', fillcolor='rgba(148,103,189,0.12)',
                             line=dict(width=0), name='95% CI'))
fig_btv.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                  annotation_text=f'Fed {MU_ANCHOR}% target')
fig_btv.update_layout(
    title=f'US VAR({VAR_LAG}) backtest — train {TRAIN_START[:4]}-{BACKTEST_END[:4]}, '
          f'forecast 2015-{CUTOFF[:4]}',
    xaxis_title='Month', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_btv.show()

Train: 1993-01-01 to 2014-12-01, n=264
Test:  2015-01-01  to 2026-02-01,  n=133
VAR(2) fitted, AIC=-12.867

VAR backtest (headline only):
  RMSE: 2.162 pp
  MAE:  1.427 pp

Comparison with prior models:
  Baseline ARIMA: RMSE=2.161, MAE=1.409
  SARIMAX-X:      RMSE=1.974, MAE=1.356
  VAR(2):          RMSE=2.162, MAE=1.427


In [28]:
# ===== VAR deliverable: refit on full sample, 10-year forecast =====
var_full = df.loc[TRAIN_START:CUTOFF,
                  ['headline_yoy', 'trimmed_mean_yoy', 'median_yoy']].dropna()

final_var = VAR(var_full).fit(VAR_LAG)
print(f'VAR({VAR_LAG}) on full sample (n={len(var_full)}), AIC={final_var.aic:.3f}')

# In-sample fitted values
var_resid  = final_var.resid
var_fitted = var_full.iloc[VAR_LAG:] - var_resid

var_h_resid = var_resid['headline_yoy']
rmse_in_v = np.sqrt(np.mean(var_h_resid**2))
mae_in_v  = np.mean(np.abs(var_h_resid))
print(f'\nVAR in-sample fit (headline): RMSE={rmse_in_v:.3f} pp, MAE={mae_in_v:.3f} pp')
print(f'  Baseline ARIMA in-sample:   RMSE={rmse_in:.3f} pp, MAE={mae_in:.3f} pp')
print(f'  SARIMAX-X in-sample:        RMSE={rmse_in_x:.3f} pp, MAE={mae_in_x:.3f} pp')

# Forecast 120 months
fcv_mid, fcv_lo95, fcv_hi95 = final_var.forecast_interval(
    y=var_full.values[-VAR_LAG:], steps=HORIZON, alpha=0.05)
_, fcv_lo80, fcv_hi80 = final_var.forecast_interval(
    y=var_full.values[-VAR_LAG:], steps=HORIZON, alpha=0.20)

fcv_idx = pd.date_range(start=var_full.index[-1] + pd.offsets.MonthEnd(),
                        periods=HORIZON, freq=FREQ)

fcv_h_mean = fcv_mid[:, H]
fcv_h_lo95 = fcv_lo95[:, H]
fcv_h_hi95 = fcv_hi95[:, H]
fcv_h_lo80 = fcv_lo80[:, H]
fcv_h_hi80 = fcv_hi80[:, H]

print(f'\nVAR 10-yr forecast: mean {fcv_h_mean.mean():.2f}%, end {fcv_h_mean[-1]:.2f}%')
print(f'\n--- Three-model comparison ---')
print(f'  Baseline ARIMA: mean={fc_mean.mean():.2f}%   target={fc_mean.mean() + 2.5:.2f}% p.a.')
print(f'  SARIMAX-X:      mean={fcx_mean.mean():.2f}%   target={fcx_mean.mean() + 2.5:.2f}% p.a.')
print(f'  VAR({VAR_LAG}):          mean={fcv_h_mean.mean():.2f}%   target={fcv_h_mean.mean() + 2.5:.2f}% p.a.')

# Plot: actual + in-sample fit + forecast + CI + comparison forecasts
fig_fcv = go.Figure()
fig_fcv.add_trace(go.Scatter(x=var_full.index, y=var_full['headline_yoy'].values,
                             mode='lines', name='Actual',
                             line=dict(color='#1f77b4', width=2)))
fig_fcv.add_trace(go.Scatter(x=var_fitted.index, y=var_fitted['headline_yoy'].values,
                             mode='lines', name='In-sample fit',
                             line=dict(color='#9467bd', width=1.5, dash='dot')))
fig_fcv.add_trace(go.Scatter(x=fcv_idx, y=fcv_h_mean, mode='lines',
                             name=f'VAR({VAR_LAG}) (forecast)',
                             line=dict(color='#9467bd', dash='dash', width=2)))
fig_fcv.add_trace(go.Scatter(x=list(fcv_idx) + list(fcv_idx[::-1]),
                             y=list(fcv_h_hi95) + list(fcv_h_lo95[::-1]),
                             fill='toself', fillcolor='rgba(148,103,189,0.10)',
                             line=dict(width=0), name='95% CI (VAR)'))
fig_fcv.add_trace(go.Scatter(x=list(fcv_idx) + list(fcv_idx[::-1]),
                             y=list(fcv_h_hi80) + list(fcv_h_lo80[::-1]),
                             fill='toself', fillcolor='rgba(148,103,189,0.20)',
                             line=dict(width=0), name='80% CI (VAR)'))
fig_fcv.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                  annotation_text=f'Fed {MU_ANCHOR}% target')
fig_fcv.update_layout(
    title=f'US VAR({VAR_LAG}) — in-sample fit + 10-year forecast (3-model comparison)',
    xaxis_title='Month', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_fcv.show()

# Save VAR forecast
forecast_var_out = pd.DataFrame({
    'yoy_mean': fcv_h_mean,
    'yoy_lo80': fcv_h_lo80,
    'yoy_hi80': fcv_h_hi80,
    'yoy_lo95': fcv_h_lo95,
    'yoy_hi95': fcv_h_hi95,
}, index=fcv_idx)
print(f'\nVAR forecast saved as `forecast_var_out`. First/last rows:')
print(forecast_var_out.head(2).round(2))
print(forecast_var_out.tail(2).round(2))

VAR(2) on full sample (n=397), AIC=-12.743

VAR in-sample fit (headline): RMSE=0.360 pp, MAE=0.261 pp
  Baseline ARIMA in-sample:   RMSE=0.248 pp, MAE=0.174 pp
  SARIMAX-X in-sample:        RMSE=0.197 pp, MAE=0.136 pp

VAR 10-yr forecast: mean 2.54%, end 2.54%

--- Three-model comparison ---
  Baseline ARIMA: mean=2.26%   target=4.76% p.a.
  SARIMAX-X:      mean=1.95%   target=4.45% p.a.
  VAR(2):          mean=2.54%   target=5.04% p.a.



VAR forecast saved as `forecast_var_out`. First/last rows:
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95
2026-02-28      2.47       2.0      2.93      1.76      3.18
2026-03-31      2.49       1.7      3.29      1.28      3.71
            yoy_mean  yoy_lo80  yoy_hi80  yoy_lo95  yoy_hi95
2035-12-31      2.54      0.54      4.53     -0.52      5.59
2036-01-31      2.54      0.54      4.53     -0.52      5.59


# State-Space with trimmed mean as latent signal

In [29]:
# ===== State-space: headline = latent signal + transitory shock =====
# Model:
#   headline_t = signal_t + ε_t           (observation)
#   signal_t   = (1-φ)·μ + φ·signal_{t-1} + η_t   (state, AR(1) toward μ)
# μ = long-run mean (anchored to Fed target = 2.0% for US)
# φ = persistence; estimated.
# Trimmed mean is our best observable proxy for underlying inflation.

from statsmodels.tsa.statespace.mlemodel import MLEModel

class LatentInflation(MLEModel):
    """AR(1) latent signal anchored to long-run mean μ, observed with noise."""
    start_params = [0.85, 0.5, 0.3]   # phi, sigma_obs, sigma_state
    param_names  = ['phi', 'sigma_obs', 'sigma_state']

    def __init__(self, endog, mu_anchor=2.0):
        super().__init__(endog, k_states=1, initialization='approximate_diffuse')
        self.mu_anchor = mu_anchor
        self['design', 0, 0]    = 1.0
        self['transition', 0, 0] = 0.0   # set in update()
        self['selection', 0, 0]  = 1.0

    def update(self, params, **kwargs):
        phi, sig_obs, sig_state = params
        self['transition', 0, 0]    = phi
        self['state_intercept', 0]  = (1 - phi) * self.mu_anchor
        self['obs_cov', 0, 0]       = sig_obs ** 2
        self['state_cov', 0, 0]     = sig_state ** 2

    def transform_params(self, p):
        return np.array([np.tanh(p[0]), np.exp(p[1]), np.exp(p[2])])
    def untransform_params(self, p):
        return np.array([np.arctanh(p[0]), np.log(p[1]), np.log(p[2])])

# --- Fit on full sample, anchor μ at 2.0% (Fed target) ---
ss_model = LatentInflation(y_full.values, mu_anchor=MU_ANCHOR)
ss_fit = ss_model.fit(disp=False, maxiter=500)
phi, sig_obs, sig_state = ss_fit.params
print(f'State-space (anchor μ={MU_ANCHOR}%):')
print(f'  φ (persistence)   = {phi:.3f}')
print(f'  σ (obs noise)     = {sig_obs:.3f}')
print(f'  σ (state innov.)  = {sig_state:.3f}')
print(f'  AIC               = {ss_fit.aic:.2f}')
print(f'  Log-likelihood    = {ss_fit.llf:.2f}')

# Smoothed state = filtered estimate of latent signal across history
smoothed_signal = pd.Series(ss_fit.smoothed_state[0], index=y_full.index)

# In-sample fitted values
ss_fitted = pd.Series(ss_fit.fittedvalues, index=y_full.index)
ss_resid  = y_full - ss_fitted
warmup_ss = SEASONAL_M     # one full year warmup for monthly data
rmse_in_ss = np.sqrt(np.mean(ss_resid.iloc[warmup_ss:]**2))
mae_in_ss  = np.mean(np.abs(ss_resid.iloc[warmup_ss:]))
print(f'\nIn-sample fit (excl. {warmup_ss} warmup): RMSE={rmse_in_ss:.3f}, MAE={mae_in_ss:.3f}')

# --- Forecast 120 months ---
fcss_idx  = pd.date_range(start=y_full.index[-1] + pd.offsets.MonthEnd(),
                          periods=HORIZON, freq=FREQ)

fcss_obj = ss_fit.get_forecast(steps=HORIZON)
fcss_mean = pd.Series(np.asarray(fcss_obj.predicted_mean).ravel(), index=fcss_idx)
fcss_ci95 = pd.DataFrame(np.asarray(fcss_obj.conf_int(alpha=0.05)),
                         index=fcss_idx, columns=['lo', 'hi'])
fcss_ci80 = pd.DataFrame(np.asarray(fcss_obj.conf_int(alpha=0.20)),
                         index=fcss_idx, columns=['lo', 'hi'])

print(f'\nState-space 10-yr forecast: mean {fcss_mean.mean():.2f}%, end {fcss_mean.iloc[-1]:.2f}%')
print(f'  Note: long-run forecast → μ_anchor ({MU_ANCHOR}%) at rate (1-φ) per month')

# Plot: actual + smoothed signal + forecast
fig_ss = go.Figure()
fig_ss.add_trace(go.Scatter(x=y_full.index, y=y_full.values, mode='lines',
                            name='Actual headline', line=dict(color='#1f77b4', width=2)))
fig_ss.add_trace(go.Scatter(x=smoothed_signal.index, y=smoothed_signal.values,
                            mode='lines', name='Smoothed latent signal',
                            line=dict(color='#2ca02c', width=2)))
fig_ss.add_trace(go.Scatter(x=df.loc[y_full.index, 'trimmed_mean_yoy'].index,
                            y=df.loc[y_full.index, 'trimmed_mean_yoy'].values,
                            mode='lines', name='Trimmed mean (proxy)',
                            line=dict(color='#ff7f0e', dash='dot', width=1.5)))
fig_ss.add_trace(go.Scatter(x=fcss_idx, y=fcss_mean.values, mode='lines',
                            name='Forecast',
                            line=dict(color='#d62728', dash='dash', width=2)))
fig_ss.add_trace(go.Scatter(x=list(fcss_idx) + list(fcss_idx[::-1]),
                            y=list(fcss_ci95['hi']) + list(fcss_ci95['lo'][::-1]),
                            fill='toself', fillcolor='rgba(214,39,40,0.10)',
                            line=dict(width=0), name='95% CI'))
fig_ss.add_trace(go.Scatter(x=list(fcss_idx) + list(fcss_idx[::-1]),
                            y=list(fcss_ci80['hi']) + list(fcss_ci80['lo'][::-1]),
                            fill='toself', fillcolor='rgba(214,39,40,0.20)',
                            line=dict(width=0), name='80% CI'))
fig_ss.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                 annotation_text=f'Fed {MU_ANCHOR}% target / μ_anchor')
fig_ss.update_layout(
    title=f'US State-space (latent signal, AR(1) anchored to μ={MU_ANCHOR}%)',
    xaxis_title='Month', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_ss.show()

forecast_ss_out = pd.DataFrame({
    'yoy_mean': fcss_mean.values,
    'yoy_lo80': fcss_ci80['lo'].values,
    'yoy_hi80': fcss_ci80['hi'].values,
    'yoy_lo95': fcss_ci95['lo'].values,
    'yoy_hi95': fcss_ci95['hi'].values,
}, index=fcss_idx)

State-space (anchor μ=2.0%):
  φ (persistence)   = 0.969
  σ (obs noise)     = 0.000
  σ (state innov.)  = 0.404
  AIC               = 427.35
  Log-likelihood    = -210.68

In-sample fit (excl. 12 warmup): RMSE=0.409, MAE=0.283

State-space 10-yr forecast: mean 2.11%, end 2.01%
  Note: long-run forecast → μ_anchor (2.0%) at rate (1-φ) per month


# Bayesian VAR

In [32]:
# ===== Bayesian VAR(2) with Minnesota prior =====
# Minnesota prior: each variable a priori follows a random walk, with own lags
# weighted higher than cross-lags. Shrinks coefficients toward parsimonious prior,
# reducing overfitting on monthly samples (n≈397, 21 free params in BVAR(2)).
# Reference: Bańbura, Giannone, Reichlin (2010).

def fit_bvar_minnesota(data, p, lam_overall=0.2, lam_cross=0.5, lam_decay=1.0):
    Y = data.values
    T, K = Y.shape
    sigma_i = np.array([data[c].diff().dropna().std() for c in data.columns])

    X_list, Y_list = [], []
    for t in range(p, T):
        X_list.append(np.concatenate([Y[t-l-1] for l in range(p)] + [[1.0]]))
        Y_list.append(Y[t])
    X = np.array(X_list)
    Yreg = np.array(Y_list)

    Xd = np.zeros((K * p + 1, K * p + 1))
    Yd = np.zeros((K * p + 1, K))

    for l in range(p):
        for i in range(K):
            row = l * K + i
            scale = lam_overall / ((l + 1) ** lam_decay)
            Xd[row, row] = sigma_i[i] / scale
            if l == 0:
                Yd[row, i] = sigma_i[i] / scale
    Xd[-1, -1] = 1e-3

    Xstar = np.vstack([X, Xd])
    Ystar = np.vstack([Yreg, Yd])

    B_post = np.linalg.lstsq(Xstar, Ystar, rcond=None)[0]
    resid  = Yreg - X @ B_post
    Sigma  = (resid.T @ resid) / (T - p)

    return B_post, Sigma, X, Yreg

def bvar_forecast(B, last_lags, steps, K, p):
    fc = np.zeros((steps, K))
    state = list(last_lags)
    for h in range(steps):
        x = np.concatenate(state[:p] + [[1.0]])
        ynext = x @ B
        fc[h] = ynext
        state = [ynext] + state
    return fc

BVAR_LAG = 2
B_post, Sigma_post, X_in, Y_in = fit_bvar_minnesota(
    var_full, p=BVAR_LAG, lam_overall=0.2, lam_cross=0.5, lam_decay=1.0)
print(f'BVAR({BVAR_LAG}) Minnesota prior fit')
print(f'  λ_overall=0.2, λ_cross=0.5, λ_decay=1.0')

fitted_in = X_in @ B_post
resid_in_h = Y_in[:, 0] - fitted_in[:, 0]
rmse_in_b = np.sqrt(np.mean(resid_in_h ** 2))
mae_in_b  = np.mean(np.abs(resid_in_h))
print(f'\nBVAR in-sample (headline): RMSE={rmse_in_b:.3f}, MAE={mae_in_b:.3f}')
print(f'  vs frequentist VAR:       RMSE={rmse_in_v:.3f}, MAE={mae_in_v:.3f}')

# --- Forecast 120 months ---
last_lags = [var_full.values[-1], var_full.values[-2]]
fc_bvar = bvar_forecast(B_post, last_lags, HORIZON, K=3, p=BVAR_LAG)

# Forecast intervals via simulation (1000 paths)
np.random.seed(42)
n_sims = 1000
sim_paths = np.zeros((n_sims, HORIZON, 3))
chol = np.linalg.cholesky(Sigma_post)
for s in range(n_sims):
    state = [var_full.values[-1].copy(), var_full.values[-2].copy()]
    for h in range(HORIZON):
        x = np.concatenate(state[:BVAR_LAG] + [[1.0]])
        shock = chol @ np.random.randn(3)
        ynext = x @ B_post + shock
        sim_paths[s, h] = ynext
        state = [ynext] + state

bvar_h_mean = fc_bvar[:, 0]
bvar_h_lo95 = np.percentile(sim_paths[:, :, 0], 2.5, axis=0)
bvar_h_hi95 = np.percentile(sim_paths[:, :, 0], 97.5, axis=0)
bvar_h_lo80 = np.percentile(sim_paths[:, :, 0], 10, axis=0)
bvar_h_hi80 = np.percentile(sim_paths[:, :, 0], 90, axis=0)

bvar_idx = pd.date_range(start=var_full.index[-1] + pd.offsets.MonthEnd(),
                         periods=HORIZON, freq=FREQ)
print(f'\nBVAR 10-yr forecast: mean {bvar_h_mean.mean():.2f}%, end {bvar_h_mean[-1]:.2f}%')

# Plot
fig_bv = go.Figure()
fig_bv.add_trace(go.Scatter(x=var_full.index, y=var_full['headline_yoy'].values,
                            mode='lines', name='Actual', line=dict(color='#1f77b4', width=2)))
fig_bv.add_trace(go.Scatter(x=bvar_idx, y=bvar_h_mean, mode='lines',
                            name=f'BVAR({BVAR_LAG}) (forecast)',
                            line=dict(color='#17becf', dash='dash', width=2)))
fig_bv.add_trace(go.Scatter(x=list(bvar_idx) + list(bvar_idx[::-1]),
                            y=list(bvar_h_hi95) + list(bvar_h_lo95[::-1]),
                            fill='toself', fillcolor='rgba(23,190,207,0.10)',
                            line=dict(width=0), name='95% CI (simulated)'))
fig_bv.add_trace(go.Scatter(x=list(bvar_idx) + list(bvar_idx[::-1]),
                            y=list(bvar_h_hi80) + list(bvar_h_lo80[::-1]),
                            fill='toself', fillcolor='rgba(23,190,207,0.20)',
                            line=dict(width=0), name='80% CI (simulated)'))
fig_bv.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                 annotation_text=f'Fed {MU_ANCHOR}% target')
fig_bv.update_layout(
    title=f'US Bayesian VAR({BVAR_LAG}) with Minnesota prior — 10-year forecast',
    xaxis_title='Month', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified')
fig_bv.show()

forecast_bvar_out = pd.DataFrame({
    'yoy_mean': bvar_h_mean,
    'yoy_lo80': bvar_h_lo80,
    'yoy_hi80': bvar_h_hi80,
    'yoy_lo95': bvar_h_lo95,
    'yoy_hi95': bvar_h_hi95,
}, index=bvar_idx)

BVAR(2) Minnesota prior fit
  λ_overall=0.2, λ_cross=0.5, λ_decay=1.0

BVAR in-sample (headline): RMSE=0.362, MAE=0.260
  vs frequentist VAR:       RMSE=0.360, MAE=0.261

BVAR 10-yr forecast: mean 2.53%, end 2.54%


# Summary

In [33]:
# ===== Five-model summary (US — no CPI+2.5% target, that's AU mandate only) =====

print('=' * 60)
print('FIVE-MODEL SUMMARY — US 10-year mean headline CPI YoY')
print('=' * 60)

models = [
    ('Baseline ARIMA',   fc_mean.mean(),    fc_mean.iloc[-1]),
    ('SARIMAX-X',        fcx_mean.mean(),   fcx_mean.iloc[-1]),
    ('VAR(2)',           fcv_h_mean.mean(), fcv_h_mean[-1]),
    ('State-space',      fcss_mean.mean(),  fcss_mean.iloc[-1]),
    ('Bayesian VAR(2)',  bvar_h_mean.mean(),bvar_h_mean[-1]),
]
for name, mean_, end_ in models:
    print(f'  {name:18s} {mean_:.2f}%')

print()

# ----- Compact comparison table -----
summary_table = pd.DataFrame({
    'Model':      [m[0] for m in models],
    'Mean YoY':   [round(m[1], 2) for m in models],
    'End horizon':[round(m[2], 2) for m in models],
    'In-sample RMSE': [round(rmse_in,    3),
                      round(rmse_in_x,  3),
                      round(rmse_in_v,  3),
                      round(rmse_in_ss, 3),
                      round(rmse_in_b,  3)],
    'In-sample MAE':  [round(mae_in,    3),
                      round(mae_in_x,  3),
                      round(mae_in_v,  3),
                      round(mae_in_ss, 3),
                      round(mae_in_b,  3)],
})
print(summary_table.to_string(index=False))

cpi_lo, cpi_hi = summary_table['Mean YoY'].min(), summary_table['Mean YoY'].max()
print(f'\nUS CPI assumption range: {cpi_lo}% to {cpi_hi}% (midpoint {(cpi_lo+cpi_hi)/2:.2f}%)')
print('\nThis US CPI forecast feeds nominal-return assumptions for global asset classes:')
print('  - Global Equities (MSCI ACWI ex-Australia, 50% hedged in MTG/LTG)')
print('  - Global Fixed Income (Bloomberg Global Aggregate, hedged)')
print('  - Global Credit (Bloomberg Global Credit, hedged)')
print('  - Global Private Equity (LPX50, 100% hedged in LTG)')
print('  - Global Infrastructure (FTSE Global Core Infrastructure, unhedged in LTG)')

FIVE-MODEL SUMMARY — US 10-year mean headline CPI YoY
  Baseline ARIMA     2.26%
  SARIMAX-X          1.95%
  VAR(2)             2.54%
  State-space        2.11%
  Bayesian VAR(2)    2.53%

          Model  Mean YoY  End horizon  In-sample RMSE  In-sample MAE
 Baseline ARIMA      2.26         2.35           0.248          0.174
      SARIMAX-X      1.95         1.69           0.197          0.136
         VAR(2)      2.54         2.54           0.360          0.261
    State-space      2.11         2.01           0.409          0.283
Bayesian VAR(2)      2.53         2.54           0.362          0.260

US CPI assumption range: 1.95% to 2.54% (midpoint 2.25%)

This US CPI forecast feeds nominal-return assumptions for global asset classes:
  - Global Equities (MSCI ACWI ex-Australia, 50% hedged in MTG/LTG)
  - Global Fixed Income (Bloomberg Global Aggregate, hedged)
  - Global Credit (Bloomberg Global Credit, hedged)
  - Global Private Equity (LPX50, 100% hedged in LTG)
  - Global Infra

In [36]:
# ===== Overlay plot: YoY in-sample fit + forecast across all 5 US models =====

# Build BVAR in-sample fitted aligned to dates
bvar_in_idx = var_full.index[BVAR_LAG:]
bvar_in_fitted = pd.Series(Y_in[:, 0] - resid_in_h, index=bvar_in_idx)

# VAR in-sample fitted
var_in_fitted = var_fitted['headline_yoy']

fig_all = go.Figure()

# Actual
fig_all.add_trace(go.Scatter(x=y_full.index, y=y_full.values, mode='lines',
                             name='Actual', line=dict(color='black', width=2.2)))

# In-sample fits — thin dotted
fig_all.add_trace(go.Scatter(x=arima_fitted.index, y=arima_fitted.values, mode='lines',
                             name='ARIMA fit',
                             line=dict(color='#ff7f0e', width=1, dash='dot'),
                             legendgroup='ARIMA'))
fig_all.add_trace(go.Scatter(x=sarimax_fitted.index, y=sarimax_fitted.values, mode='lines',
                             name='SARIMAX-X fit',
                             line=dict(color='#d62728', width=1, dash='dot'),
                             legendgroup='SARIMAX-X'))
fig_all.add_trace(go.Scatter(x=var_in_fitted.index, y=var_in_fitted.values, mode='lines',
                             name='VAR fit',
                             line=dict(color='#9467bd', width=1, dash='dot'),
                             legendgroup='VAR'))
fig_all.add_trace(go.Scatter(x=ss_fitted.index, y=ss_fitted.values, mode='lines',
                             name='State-space fit',
                             line=dict(color='#2ca02c', width=1, dash='dot'),
                             legendgroup='SS'))
fig_all.add_trace(go.Scatter(x=bvar_in_fitted.index, y=bvar_in_fitted.values, mode='lines',
                             name='BVAR fit',
                             line=dict(color='#17becf', width=1, dash='dot'),
                             legendgroup='BVAR'))

# Forecasts — thicker dashed
fig_all.add_trace(go.Scatter(x=fc_idx, y=fc_mean.values, mode='lines',
                             name='ARIMA forecast',
                             line=dict(color='#ff7f0e', width=2.5, dash='dash'),
                             legendgroup='ARIMA'))
fig_all.add_trace(go.Scatter(x=fcx_idx, y=fcx_mean.values, mode='lines',
                             name='SARIMAX-X forecast',
                             line=dict(color='#d62728', width=2.5, dash='dash'),
                             legendgroup='SARIMAX-X'))
fig_all.add_trace(go.Scatter(x=fcv_idx, y=fcv_h_mean, mode='lines',
                             name='VAR forecast',
                             line=dict(color='#9467bd', width=2.5, dash='dash'),
                             legendgroup='VAR'))
fig_all.add_trace(go.Scatter(x=fcss_idx, y=fcss_mean.values, mode='lines',
                             name='State-space forecast',
                             line=dict(color='#2ca02c', width=2.5, dash='dash'),
                             legendgroup='SS'))
fig_all.add_trace(go.Scatter(x=bvar_idx, y=bvar_h_mean, mode='lines',
                             name='BVAR forecast',
                             line=dict(color='#17becf', width=2.5, dash='dash'),
                             legendgroup='BVAR'))

fig_all.add_hline(y=MU_ANCHOR, line_dash='dot', line_color='grey',
                  annotation_text=f'Fed {MU_ANCHOR}% target',
                  annotation_position='top right')
fig_all.update_layout(
    title='US — All five models, in-sample fit + 10-year CPI YoY forecast',
    xaxis_title='Month', yaxis_title='Headline CPI YoY (%)',
    template='plotly_white', hovermode='x unified',
    legend=dict(x=1.02, y=1.0))
fig_all.show()


# ===== Implied CPI index level path =====
# Monthly: YoY uses 12-month lookback, not 4

def yoy_to_level(yoy_array, hist_index, lookback=12):
    level_path = list(hist_index.iloc[-lookback:].values)
    for yoy in yoy_array:
        level_path.append(level_path[-lookback] * (1 + yoy / 100))
    return level_path[lookback:]

hist_idx = df['cpi_index'].dropna()

level_arima = yoy_to_level(fc_mean.values,    hist_idx)
level_sx    = yoy_to_level(fcx_mean.values,   hist_idx)
level_var   = yoy_to_level(fcv_h_mean,        hist_idx)
level_ss    = yoy_to_level(fcss_mean.values,  hist_idx)
level_bvar  = yoy_to_level(bvar_h_mean,       hist_idx)

fig_lvl_all = go.Figure()
fig_lvl_all.add_trace(go.Scatter(x=hist_idx.index, y=hist_idx.values, mode='lines',
                                 name='Historical CPI',
                                 line=dict(color='black', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=fc_idx, y=level_arima, mode='lines',
                                 name='ARIMA',
                                 line=dict(color='#ff7f0e', dash='dash', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=fcx_idx, y=level_sx, mode='lines',
                                 name='SARIMAX-X',
                                 line=dict(color='#d62728', dash='dash', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=fcv_idx, y=level_var, mode='lines',
                                 name='VAR',
                                 line=dict(color='#9467bd', dash='dash', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=fcss_idx, y=level_ss, mode='lines',
                                 name='State-space',
                                 line=dict(color='#2ca02c', dash='dash', width=2)))
fig_lvl_all.add_trace(go.Scatter(x=bvar_idx, y=level_bvar, mode='lines',
                                 name='BVAR',
                                 line=dict(color='#17becf', dash='dash', width=2)))
fig_lvl_all.update_layout(
    title='US Implied CPI Index Path — five-model comparison (1982-84 = 100)',
    xaxis_title='Month', yaxis_title='Index level',
    template='plotly_white', hovermode='x unified',
    legend=dict(x=1.02, y=1.0))
fig_lvl_all.show()

# 10-year cumulative inflation
last_idx = hist_idx.iloc[-1]
last_dt  = hist_idx.index[-1].date()
print(f'\n10-year cumulative US CPI level (start = {last_idx:.2f}, {last_dt}):')
for name, lvl in [('ARIMA', level_arima), ('SARIMAX-X', level_sx),
                  ('VAR', level_var), ('State-space', level_ss),
                  ('BVAR', level_bvar)]:
    cum = (lvl[-1] / last_idx - 1) * 100
    print(f'  {name:14s}  end-of-horizon index = {lvl[-1]:.2f}  '
          f'cumulative inflation = {cum:.1f}%')


10-year cumulative US CPI level (start = 327.46, 2026-02-01):
  ARIMA           end-of-horizon index = 424.03  cumulative inflation = 29.5%
  SARIMAX-X       end-of-horizon index = 394.85  cumulative inflation = 20.6%
  VAR             end-of-horizon index = 420.84  cumulative inflation = 28.5%
  State-space     end-of-horizon index = 402.80  cumulative inflation = 23.0%
  BVAR            end-of-horizon index = 420.50  cumulative inflation = 28.4%
